## 병렬처리 이전 코드

In [1]:
import os
import re
import time
import random
import pandas as pd
from openai import OpenAI

# =========================================================
# 0. OpenAI client
# =========================================================
client = OpenAI(api_key=os.getenv("OPENAI_API_KEY"))

In [2]:
# =========================================================
# 1. 기본 설정
# =========================================================
MODEL_NAME = "gpt-5.4"   # 사용 가능한 모델명으로 변경 가능
RANDOM_STATE = 42
random.seed(RANDOM_STATE)

TARGET_PER_TOPIC = 10_000
BATCH_DOCS = 20
SAVE_EVERY = 5

OUTPUT_DIR = "../data/processed/ai"
os.makedirs(OUTPUT_DIR, exist_ok=True)

topic_info = {
    "320": {
        "topic_name": "경제학",
        "prompt_topic": "경제학 개념, 시장, 정책, 소비, 생산, 분배, 금융, 경기 등과 관련된 설명형 글"
    },
    "360": {
        "topic_name": "법률·법학",
        "prompt_topic": "법, 제도, 권리, 의무, 판례, 책임, 절차, 규범 등과 관련된 설명형 글"
    },
    "370": {
        "topic_name": "교육학",
        "prompt_topic": "학습, 교육과정, 교수법, 평가, 학교, 발달, 교육환경 등과 관련된 설명형 글"
    },
    "400": {
        "topic_name": "자연과학",
        "prompt_topic": "자연 현상, 생명, 물질, 에너지, 실험, 관찰, 과학 원리 등과 관련된 설명형 글"
    },
    "440": {
        "topic_name": "천문학",
        "prompt_topic": "우주, 행성, 별, 은하, 중력, 천체 관측, 우주 현상 등과 관련된 설명형 글"
    },
}

STYLE_POOL = [
    "개념 설명 중심",
    "원리 설명 중심",
    "사례를 곁들인 설명",
    "비교·대조 중심",
    "입문서 스타일의 설명",
]

LENGTH_POOL = [
    "4~6문장",
    "5~7문장",
    "6~8문장",
]

TEMPERATURE_POOL = [0.7, 0.9, 1.0]

In [3]:
# =========================================================
# 2. 텍스트 처리 함수
# =========================================================

def normalize_text(text):
    if text is None:
        return ""
    text = str(text).strip()
    text = re.sub(r"\s+", " ", text)
    text = text.replace("“", '"').replace("”", '"').replace("’", "'").replace("‘", "'")
    return text

def split_sentences_korean(text):
    text = normalize_text(text)
    text = text.replace("\n", " ")
    sents = re.split(r'(?<=[\.\?\!])\s+', text)

    if len(sents) == 1:
        sents = re.split(r'(?<=다)\s+|(?<=요)\s+|(?<=니다)\s+', text)

    return [s.strip() for s in sents if s.strip()]

def is_valid_sentence(text, min_words=5, max_words=50):
    text = normalize_text(text)

    if not text:
        return False
    if len(text) < 10:
        return False
    if not re.search(r"[가-힣]", text):
        return False

    word_count = len(text.split())
    if word_count < min_words or word_count > max_words:
        return False

    hangul_chars = re.findall(r"[가-힣]", text)
    if len(hangul_chars) / max(len(text), 1) < 0.3:
        return False

    meta_patterns = [
        r"^제\s*\d+\s*장",
        r"^제\s*\d+\s*절",
        r"^\d+\.$",
        r"^\d+\)",
        r"^부록",
        r"^참고문헌",
        r"^목차",
        r"^저자",
        r"^출판사",
        r"^ISBN",
        r"^머리말",
        r"^서문",
        r"^차례",
    ]
    for pattern in meta_patterns:
        if re.search(pattern, text):
            return False

    return True

def clean_ai_sentences(sentences):
    cleaned = []
    seen = set()

    for sent in sentences:
        sent = normalize_text(sent)
        sent = re.sub(r'^\s*[-•\d\.\)\(]+\s*', '', sent).strip()

        if is_valid_sentence(sent):
            if sent not in seen:
                cleaned.append(sent)
                seen.add(sent)

    return cleaned

In [4]:
# =========================================================
# 3. 프롬프트 및 생성 함수
# =========================================================

def make_generation_prompt(topic_name, topic_desc, style, length_spec, batch_docs):
    prompt = f"""
다음 조건에 맞는 한국어 설명문 여러 개를 작성하시오.

[분야]
{topic_name}

[세부 주제 범위]
{topic_desc}

[문체]
- 자연스러운 설명문 형태
- 지나치게 문학적이거나 대화체인 문장은 피할 것
- 불필요한 제목이나 메모는 쓰지 말 것
- 동일 표현 반복을 피할 것

[스타일]
{style}

[길이]
각 글은 {length_spec}

[출력 형식]
- 서로 다른 짧은 글 {batch_docs}개를 생성할 것
- 각 글은 1, 2, 3 ... 번호를 붙여 구분할 것
- 각 글은 하나의 짧은 문단 형태로 작성할 것
- 반드시 한국어만 사용할 것
"""
    return prompt.strip()

def generate_batch_text(topic_name, topic_desc, batch_docs=BATCH_DOCS):
    style = random.choice(STYLE_POOL)
    length_spec = random.choice(LENGTH_POOL)
    temperature = random.choice(TEMPERATURE_POOL)

    prompt = make_generation_prompt(
        topic_name=topic_name,
        topic_desc=topic_desc,
        style=style,
        length_spec=length_spec,
        batch_docs=batch_docs
    )

    response = client.responses.create(
        model=MODEL_NAME,
        input=prompt,
        temperature=temperature
    )

    text = getattr(response, "output_text", None)
    if not text:
        raise ValueError("응답에서 output_text를 찾지 못했다.")

    return text, {
        "style": style,
        "length_spec": length_spec,
        "temperature": temperature
    }

def split_generated_documents(raw_text):
    raw_text = normalize_text(raw_text)
    parts = re.split(r'\s*(?=\d+\.\s)', raw_text)

    docs = []
    for part in parts:
        part = part.strip()
        if not part:
            continue
        part = re.sub(r'^\d+\.\s*', '', part).strip()
        if part:
            docs.append(part)

    if len(docs) <= 1:
        docs = [x.strip() for x in raw_text.split("\n") if x.strip()]

    return docs

In [5]:
# =========================================================
# 4. 저장 파일명 함수
# =========================================================

def get_temp_save_path(topic_code, batch_num):
    return os.path.join(OUTPUT_DIR, f"ai_{topic_code}_temp_until_batch_{batch_num}.csv")

def get_final_save_path(topic_code, target_n):
    if target_n == 10_000:
        return os.path.join(OUTPUT_DIR, f"ai_{topic_code}_sampled_100k.csv")
    return os.path.join(OUTPUT_DIR, f"ai_{topic_code}_sampled_{target_n}.csv")

In [6]:
# =========================================================
# 5. 주제 하나 실행
# =========================================================

def build_ai_topic_dataset(topic_code, target_n=10_000, save_every=5, batch_docs=BATCH_DOCS):
    topic_name = topic_info[topic_code]["topic_name"]
    topic_desc = topic_info[topic_code]["prompt_topic"]

    print(f"\n===== {topic_name} ({topic_code}) AI 데이터 생성 시작 =====")

    all_rows = []
    seen_sentences = set()
    batch_num = 0

    while len(all_rows) < target_n:
        batch_num += 1

        try:
            raw_text, meta = generate_batch_text(topic_name, topic_desc, batch_docs=batch_docs)
            docs = split_generated_documents(raw_text)

            batch_rows = []

            for doc_idx, doc in enumerate(docs, start=1):
                sentences = split_sentences_korean(doc)
                cleaned_sents = clean_ai_sentences(sentences)

                for sent in cleaned_sents:
                    if sent not in seen_sentences:
                        batch_rows.append({
                            "text": sent,
                            "topic_code": topic_code,
                            "topic_name": topic_name,
                            "source": "ai",
                            "generation_style": meta["style"],
                            "length_spec": meta["length_spec"],
                            "temperature": meta["temperature"],
                            "batch_num": batch_num,
                            "doc_in_batch": doc_idx,
                            "word_count": len(sent.split())
                        })
                        seen_sentences.add(sent)

            if batch_rows:
                all_rows.extend(batch_rows)

            print(f"[{topic_name}] batch={batch_num} | 누적 문장 수={len(all_rows):,}")

            if batch_num % save_every == 0:
                temp_df = pd.DataFrame(all_rows)
                temp_path = get_temp_save_path(topic_code, batch_num)
                temp_df.to_csv(temp_path, index=False, encoding="utf-8-sig")
                print(f"중간 저장 완료 -> {temp_path}")

            time.sleep(0.8)

        except Exception as e:
            print(f"[오류] batch={batch_num}, error={e}")
            time.sleep(3)

    final_df = pd.DataFrame(all_rows).head(target_n).copy()
    final_df = final_df.drop_duplicates(subset=["text"]).reset_index(drop=True)

    while len(final_df) < target_n:
        needed = target_n - len(final_df)
        print(f"[{topic_name}] 중복 제거 후 부족분 {needed:,}개 추가 생성")

        try:
            raw_text, meta = generate_batch_text(topic_name, topic_desc, batch_docs=batch_docs)
            docs = split_generated_documents(raw_text)

            new_rows = []
            existing = set(final_df["text"].tolist())

            for doc_idx, doc in enumerate(docs, start=1):
                sentences = split_sentences_korean(doc)
                cleaned_sents = clean_ai_sentences(sentences)

                for sent in cleaned_sents:
                    if sent not in existing:
                        new_rows.append({
                            "text": sent,
                            "topic_code": topic_code,
                            "topic_name": topic_name,
                            "source": "ai",
                            "generation_style": meta["style"],
                            "length_spec": meta["length_spec"],
                            "temperature": meta["temperature"],
                            "batch_num": batch_num,
                            "doc_in_batch": doc_idx,
                            "word_count": len(sent.split())
                        })
                        existing.add(sent)

            if new_rows:
                final_df = pd.concat([final_df, pd.DataFrame(new_rows)], axis=0).reset_index(drop=True)
                final_df = final_df.drop_duplicates(subset=["text"]).reset_index(drop=True)

            time.sleep(0.8)

        except Exception as e:
            print(f"[부족분 생성 오류] {e}")
            time.sleep(3)

    final_df = final_df.head(target_n).copy()

    final_path = get_final_save_path(topic_code, target_n)
    final_df.to_csv(final_path, index=False, encoding="utf-8-sig")

    print(f"===== {topic_name} 최종 완료: {len(final_df):,}문장 =====")
    print(f"최종 저장 완료 -> {final_path}")

    return final_df

### 경제학(320)

In [7]:
# =========================================================
# 6. 실행 예시
# =========================================================

# 경제학만 실행
df_ai_320 = build_ai_topic_dataset(topic_code="320", target_n=100_000, save_every=5)


===== 경제학 (320) AI 데이터 생성 시작 =====
[경제학] batch=1 | 누적 문장 수=80
[경제학] batch=2 | 누적 문장 수=159
[경제학] batch=3 | 누적 문장 수=278
[경제학] batch=4 | 누적 문장 수=357
[경제학] batch=5 | 누적 문장 수=437
중간 저장 완료 -> ../data/processed/ai\ai_320_temp_until_batch_5.csv
[경제학] batch=6 | 누적 문장 수=512
[경제학] batch=7 | 누적 문장 수=629
[경제학] batch=8 | 누적 문장 수=706
[경제학] batch=9 | 누적 문장 수=799
[경제학] batch=10 | 누적 문장 수=939
중간 저장 완료 -> ../data/processed/ai\ai_320_temp_until_batch_10.csv
[경제학] batch=11 | 누적 문장 수=1,015
[경제학] batch=12 | 누적 문장 수=1,115
[경제학] batch=13 | 누적 문장 수=1,194
[경제학] batch=14 | 누적 문장 수=1,264
[경제학] batch=15 | 누적 문장 수=1,359
중간 저장 완료 -> ../data/processed/ai\ai_320_temp_until_batch_15.csv
[경제학] batch=16 | 누적 문장 수=1,446
[경제학] batch=17 | 누적 문장 수=1,565
[경제학] batch=18 | 누적 문장 수=1,641
[경제학] batch=19 | 누적 문장 수=1,759
[경제학] batch=20 | 누적 문장 수=1,874
중간 저장 완료 -> ../data/processed/ai\ai_320_temp_until_batch_20.csv
[경제학] batch=21 | 누적 문장 수=1,949
[경제학] batch=22 | 누적 문장 수=2,064
[경제학] batch=23 | 누적 문장 수=2,137
[경제학] batch=24 | 누적 문장 수=2

KeyboardInterrupt: 

In [8]:
import os
import re
import glob
import pandas as pd

# ==============================
# 설정
# ==============================
DATA_DIR = "../data/processed/ai"   # 네 폴더 경로로 수정
TOPIC_CODE = "320"
TARGET_N = 10000

# ==============================
# 가장 최신 temp csv 찾기
# ==============================
pattern = os.path.join(DATA_DIR, f"ai_{TOPIC_CODE}_temp_until_batch_*.csv")
file_list = glob.glob(pattern)

if not file_list:
    raise FileNotFoundError(f"해당 패턴의 파일이 없음: {pattern}")

def extract_batch_num(filepath):
    filename = os.path.basename(filepath)
    match = re.search(rf"ai_{TOPIC_CODE}_temp_until_batch_(\d+)\.csv$", filename)
    return int(match.group(1)) if match else -1

latest_file = max(file_list, key=extract_batch_num)
latest_batch = extract_batch_num(latest_file)

print(f"가장 최신 temp 파일: {latest_file}")
print(f"batch 번호: {latest_batch}")

# ==============================
# 파일 읽기
# ==============================
df = pd.read_csv(latest_file)

print(f"원본 행 개수: {len(df)}")
print(f"컬럼명: {df.columns.tolist()}")

# ==============================
# 1만 개만 자르기
# ==============================
df_10000 = df.iloc[:TARGET_N].copy()

print(f"최종 사용할 행 개수: {len(df_10000)}")

# ==============================
# 저장
# ==============================
save_path = os.path.join(DATA_DIR, f"ai_{TOPIC_CODE}_final_{TARGET_N}.csv")
df_10000.to_csv(save_path, index=False, encoding="utf-8-sig")

print(f"저장 완료: {save_path}")

가장 최신 temp 파일: ../data/processed/ai\ai_320_temp_until_batch_290.csv
batch 번호: 290
원본 행 개수: 25444
컬럼명: ['text', 'topic_code', 'topic_name', 'source', 'generation_style', 'length_spec', 'temperature', 'batch_num', 'doc_in_batch', 'word_count']
최종 사용할 행 개수: 10000
저장 완료: ../data/processed/ai\ai_320_final_10000.csv


In [9]:
import os
import re
import glob
import pandas as pd

DATA_DIR = "../data/processed/ai"
TOPIC_CODE = "320"
TARGET_N = 10000

pattern = os.path.join(DATA_DIR, f"ai_{TOPIC_CODE}_temp_until_batch_*.csv")
file_list = glob.glob(pattern)

if not file_list:
    raise FileNotFoundError(f"파일이 없음: {pattern}")

def extract_batch_num(filepath):
    filename = os.path.basename(filepath)
    match = re.search(rf"ai_{TOPIC_CODE}_temp_until_batch_(\d+)\.csv$", filename)
    return int(match.group(1)) if match else -1

latest_file = max(file_list, key=extract_batch_num)
df = pd.read_csv(latest_file)

n = len(df)
print(f"최신 파일: {latest_file}")
print(f"현재 데이터 개수: {n}")

if n < TARGET_N:
    print(f"경고: 현재 데이터가 {TARGET_N}개보다 적어서 {n}개만 저장함.")
    df_final = df.copy()
else:
    df_final = df.iloc[:TARGET_N].copy()

save_path = os.path.join(DATA_DIR, f"ai_{TOPIC_CODE}_final_{len(df_final)}.csv")
df_final.to_csv(save_path, index=False, encoding="utf-8-sig")

print(f"저장 완료: {save_path}")

최신 파일: ../data/processed/ai\ai_320_temp_until_batch_290.csv
현재 데이터 개수: 25444
저장 완료: ../data/processed/ai\ai_320_final_10000.csv


In [11]:
# print(df_final.head())
print(df_final.isnull().sum())
print("중복 행 수:", df_final.duplicated().sum())

text                0
topic_code          0
topic_name          0
source              0
generation_style    0
length_spec         0
temperature         0
batch_num           0
doc_in_batch        0
word_count          0
dtype: int64
중복 행 수: 0


## 병렬처리 이후 코드

In [10]:
import os
import re
import time
import random
import pandas as pd
from openai import OpenAI
from concurrent.futures import ThreadPoolExecutor, as_completed

# =========================================================
# 0. OpenAI client
# =========================================================
client = OpenAI(api_key=os.getenv("OPENAI_API_KEY"))

In [11]:
# =========================================================
# 1. 기본 설정
# =========================================================
MODEL_NAME = "gpt-5.4-mini"
RANDOM_STATE = 42
random.seed(RANDOM_STATE)

TARGET_PER_TOPIC = 10_000
BATCH_DOCS = 20
SAVE_EVERY = 5

OUTPUT_DIR = "../data/processed/ai"
os.makedirs(OUTPUT_DIR, exist_ok=True)

# 병렬 처리 수
MAX_WORKERS = 4   # 처음엔 4로 시작, 괜찮으면 6 정도까지 시도

topic_info = {
    "320": {
        "topic_name": "경제학",
        "prompt_topic": "경제학 개념, 시장, 정책, 소비, 생산, 분배, 금융, 경기 등과 관련된 설명형 글"
    },
    "360": {
        "topic_name": "법률·법학",
        "prompt_topic": "법, 제도, 권리, 의무, 판례, 책임, 절차, 규범 등과 관련된 설명형 글"
    },
    "370": {
        "topic_name": "교육학",
        "prompt_topic": "학습, 교육과정, 교수법, 평가, 학교, 발달, 교육환경 등과 관련된 설명형 글"
    },
    "400": {
        "topic_name": "자연과학",
        "prompt_topic": "자연 현상, 생명, 물질, 에너지, 실험, 관찰, 과학 원리 등과 관련된 설명형 글"
    },
    "440": {
        "topic_name": "천문학",
        "prompt_topic": "우주, 행성, 별, 은하, 중력, 천체 관측, 우주 현상 등과 관련된 설명형 글"
    },
}

STYLE_POOL = [
    "개념 설명 중심",
    "원리 설명 중심",
    "사례를 곁들인 설명",
    "비교·대조 중심",
    "입문서 스타일의 설명",
]

LENGTH_POOL = [
    "4~6문장",
    "5~7문장",
    "6~8문장",
]

TEMPERATURE_POOL = [0.7, 0.9, 1.0]

In [12]:
# =========================================================
# 2. 텍스트 처리 함수
# =========================================================

def normalize_text(text):
    if text is None:
        return ""
    text = str(text).strip()
    text = re.sub(r"\s+", " ", text)
    text = text.replace("“", '"').replace("”", '"').replace("’", "'").replace("‘", "'")
    return text

def split_sentences_korean(text):
    text = normalize_text(text)
    text = text.replace("\n", " ")
    sents = re.split(r'(?<=[\.\?\!])\s+', text)

    if len(sents) == 1:
        sents = re.split(r'(?<=다)\s+|(?<=요)\s+|(?<=니다)\s+', text)

    return [s.strip() for s in sents if s.strip()]

def is_valid_sentence(text, min_words=5, max_words=50):
    text = normalize_text(text)

    if not text:
        return False
    if len(text) < 10:
        return False
    if not re.search(r"[가-힣]", text):
        return False

    word_count = len(text.split())
    if word_count < min_words or word_count > max_words:
        return False

    hangul_chars = re.findall(r"[가-힣]", text)
    if len(hangul_chars) / max(len(text), 1) < 0.3:
        return False

    meta_patterns = [
        r"^제\s*\d+\s*장",
        r"^제\s*\d+\s*절",
        r"^\d+\.$",
        r"^\d+\)",
        r"^부록",
        r"^참고문헌",
        r"^목차",
        r"^저자",
        r"^출판사",
        r"^ISBN",
        r"^머리말",
        r"^서문",
        r"^차례",
    ]
    for pattern in meta_patterns:
        if re.search(pattern, text):
            return False

    return True

def clean_ai_sentences(sentences):
    cleaned = []
    seen = set()

    for sent in sentences:
        sent = normalize_text(sent)
        sent = re.sub(r'^\s*[-•\d\.\)\(]+\s*', '', sent).strip()

        if is_valid_sentence(sent):
            if sent not in seen:
                cleaned.append(sent)
                seen.add(sent)

    return cleaned

In [13]:
# =========================================================
# 3. 프롬프트 및 생성 함수
# =========================================================

def make_generation_prompt(topic_name, topic_desc, style, length_spec, batch_docs):
    prompt = f"""
다음 조건에 맞는 한국어 설명문 여러 개를 작성하시오.

[분야]
{topic_name}

[세부 주제 범위]
{topic_desc}

[문체]
- 자연스러운 설명문 형태
- 지나치게 문학적이거나 대화체인 문장은 피할 것
- 불필요한 제목이나 메모는 쓰지 말 것
- 동일 표현 반복을 피할 것

[스타일]
{style}

[길이]
각 글은 {length_spec}

[출력 형식]
- 서로 다른 짧은 글 {batch_docs}개를 생성할 것
- 각 글은 1, 2, 3 ... 번호를 붙여 구분할 것
- 각 글은 하나의 짧은 문단 형태로 작성할 것
- 반드시 한국어만 사용할 것
"""
    return prompt.strip()

def generate_batch_text(topic_name, topic_desc, batch_docs=BATCH_DOCS):
    style = random.choice(STYLE_POOL)
    length_spec = random.choice(LENGTH_POOL)
    temperature = random.choice(TEMPERATURE_POOL)

    prompt = make_generation_prompt(
        topic_name=topic_name,
        topic_desc=topic_desc,
        style=style,
        length_spec=length_spec,
        batch_docs=batch_docs
    )

    response = client.responses.create(
        model=MODEL_NAME,
        input=prompt,
        temperature=temperature
    )

    text = getattr(response, "output_text", None)
    if not text:
        raise ValueError("응답에서 output_text를 찾지 못했다.")

    return text, {
        "style": style,
        "length_spec": length_spec,
        "temperature": temperature
    }

def split_generated_documents(raw_text):
    raw_text = normalize_text(raw_text)
    parts = re.split(r'\s*(?=\d+\.\s)', raw_text)

    docs = []
    for part in parts:
        part = part.strip()
        if not part:
            continue
        part = re.sub(r'^\d+\.\s*', '', part).strip()
        if part:
            docs.append(part)

    if len(docs) <= 1:
        docs = [x.strip() for x in raw_text.split("\n") if x.strip()]

    return docs

In [14]:
# =========================================================
# 4. 저장 파일명 함수
# =========================================================

def get_temp_save_path(topic_code, batch_num):
    return os.path.join(OUTPUT_DIR, f"ai_{topic_code}_temp_until_batch_{batch_num}.csv")

def get_final_save_path(topic_code, target_n):
    return os.path.join(OUTPUT_DIR, f"ai_{topic_code}_final_{target_n}.csv")

In [15]:
# =========================================================
# 5. 배치 1개 처리 함수 (병렬 실행용)
# =========================================================

def process_single_batch(topic_code, batch_num, batch_docs=BATCH_DOCS, max_retries=3):
    topic_name = topic_info[topic_code]["topic_name"]
    topic_desc = topic_info[topic_code]["prompt_topic"]

    for attempt in range(1, max_retries + 1):
        try:
            raw_text, meta = generate_batch_text(topic_name, topic_desc, batch_docs=batch_docs)
            docs = split_generated_documents(raw_text)

            batch_rows = []

            for doc_idx, doc in enumerate(docs, start=1):
                sentences = split_sentences_korean(doc)
                cleaned_sents = clean_ai_sentences(sentences)

                for sent in cleaned_sents:
                    batch_rows.append({
                        "text": sent,
                        "topic_code": topic_code,
                        "topic_name": topic_name,
                        "source": "ai",
                        "generation_style": meta["style"],
                        "length_spec": meta["length_spec"],
                        "temperature": meta["temperature"],
                        "batch_num": batch_num,
                        "doc_in_batch": doc_idx,
                        "word_count": len(sent.split())
                    })

            return batch_num, batch_rows, None

        except Exception as e:
            if attempt == max_retries:
                return batch_num, [], str(e)
            time.sleep(2 * attempt)

In [16]:
# =========================================================
# 6. 주제 하나 실행 (병렬 버전)
# =========================================================

def build_ai_topic_dataset_parallel(topic_code, target_n=10_000, save_every=5, batch_docs=BATCH_DOCS, max_workers=4):
    topic_name = topic_info[topic_code]["topic_name"]

    print(f"\n===== {topic_name} ({topic_code}) AI 데이터 생성 시작 =====")

    all_rows = []
    seen_sentences = set()

    submitted_batch_num = 0
    completed_batch_num = 0

    futures = {}

    with ThreadPoolExecutor(max_workers=max_workers) as executor:
        while len(all_rows) < target_n:
            while len(futures) < max_workers:
                submitted_batch_num += 1
                future = executor.submit(process_single_batch, topic_code, submitted_batch_num, batch_docs)
                futures[future] = submitted_batch_num

            for future in as_completed(list(futures.keys())):
                batch_num = futures.pop(future)

                try:
                    returned_batch_num, batch_rows, error_msg = future.result()

                    if error_msg is not None:
                        print(f"[오류] {topic_name} batch={returned_batch_num}, error={error_msg}")
                    else:
                        new_count = 0
                        for row in batch_rows:
                            sent = row["text"]
                            if sent not in seen_sentences:
                                all_rows.append(row)
                                seen_sentences.add(sent)
                                new_count += 1

                        completed_batch_num += 1
                        print(f"[{topic_name}] batch={returned_batch_num} | 추가 문장 수={new_count} | 누적 문장 수={len(all_rows):,}")

                        if completed_batch_num % save_every == 0:
                            temp_df = pd.DataFrame(all_rows)
                            temp_path = get_temp_save_path(topic_code, completed_batch_num)
                            temp_df.to_csv(temp_path, index=False, encoding="utf-8-sig")
                            print(f"중간 저장 완료 -> {temp_path}")

                    if len(all_rows) >= target_n:
                        break

                    submitted_batch_num += 1
                    new_future = executor.submit(process_single_batch, topic_code, submitted_batch_num, batch_docs)
                    futures[new_future] = submitted_batch_num

                except Exception as e:
                    print(f"[예상치 못한 오류] batch={batch_num}, error={e}")

                if len(all_rows) >= target_n:
                    break

    final_df = pd.DataFrame(all_rows).drop_duplicates(subset=["text"]).reset_index(drop=True)
    final_df = final_df.head(target_n).copy()

    final_path = get_final_save_path(topic_code, target_n)
    final_df.to_csv(final_path, index=False, encoding="utf-8-sig")

    print(f"===== {topic_name} 최종 완료: {len(final_df):,}문장 =====")
    print(f"최종 저장 완료 -> {final_path}")

    return final_df

In [17]:
# =========================================================
# 7. 실행 (주제별 개별 실행)
# =========================================================

In [18]:
# 360: 법률·법학
df_ai_360 = build_ai_topic_dataset_parallel(
    topic_code="360",
    target_n=TARGET_PER_TOPIC,
    save_every=SAVE_EVERY,
    batch_docs=BATCH_DOCS,
    max_workers=MAX_WORKERS
)


===== 법률·법학 (360) AI 데이터 생성 시작 =====
[법률·법학] batch=1 | 추가 문장 수=80 | 누적 문장 수=80
[법률·법학] batch=3 | 추가 문장 수=120 | 누적 문장 수=200
[법률·법학] batch=2 | 추가 문장 수=79 | 누적 문장 수=279
[법률·법학] batch=4 | 추가 문장 수=78 | 누적 문장 수=357
[법률·법학] batch=5 | 추가 문장 수=80 | 누적 문장 수=437
중간 저장 완료 -> ../data/processed/ai\ai_360_temp_until_batch_5.csv
[법률·법학] batch=7 | 추가 문장 수=119 | 누적 문장 수=556
[법률·법학] batch=6 | 추가 문장 수=119 | 누적 문장 수=675
[법률·법학] batch=8 | 추가 문장 수=98 | 누적 문장 수=773
[법률·법학] batch=10 | 추가 문장 수=79 | 누적 문장 수=852
[법률·법학] batch=11 | 추가 문장 수=100 | 누적 문장 수=952
중간 저장 완료 -> ../data/processed/ai\ai_360_temp_until_batch_10.csv
[법률·법학] batch=9 | 추가 문장 수=120 | 누적 문장 수=1,072
[법률·법학] batch=12 | 추가 문장 수=77 | 누적 문장 수=1,149
[법률·법학] batch=13 | 추가 문장 수=79 | 누적 문장 수=1,228
[법률·법학] batch=15 | 추가 문장 수=119 | 누적 문장 수=1,347
[법률·법학] batch=14 | 추가 문장 수=119 | 누적 문장 수=1,466
중간 저장 완료 -> ../data/processed/ai\ai_360_temp_until_batch_15.csv
[법률·법학] batch=16 | 추가 문장 수=76 | 누적 문장 수=1,542
[법률·법학] batch=17 | 추가 문장 수=120 | 누적 문장 수=1,662
[법률·법학] bat

In [19]:
# 370: 교육학
df_ai_370 = build_ai_topic_dataset_parallel(
    topic_code="370",
    target_n=TARGET_PER_TOPIC,
    save_every=SAVE_EVERY,
    batch_docs=BATCH_DOCS,
    max_workers=MAX_WORKERS
)


===== 교육학 (370) AI 데이터 생성 시작 =====
[교육학] batch=3 | 추가 문장 수=80 | 누적 문장 수=80
[교육학] batch=2 | 추가 문장 수=80 | 누적 문장 수=160
[교육학] batch=1 | 추가 문장 수=80 | 누적 문장 수=240
[교육학] batch=4 | 추가 문장 수=100 | 누적 문장 수=340
[교육학] batch=5 | 추가 문장 수=80 | 누적 문장 수=420
중간 저장 완료 -> ../data/processed/ai\ai_370_temp_until_batch_5.csv
[교육학] batch=6 | 추가 문장 수=79 | 누적 문장 수=499
[교육학] batch=7 | 추가 문장 수=99 | 누적 문장 수=598
[교육학] batch=8 | 추가 문장 수=119 | 누적 문장 수=717
[교육학] batch=10 | 추가 문장 수=80 | 누적 문장 수=797
[교육학] batch=9 | 추가 문장 수=99 | 누적 문장 수=896
중간 저장 완료 -> ../data/processed/ai\ai_370_temp_until_batch_10.csv
[교육학] batch=11 | 추가 문장 수=81 | 누적 문장 수=977
[교육학] batch=12 | 추가 문장 수=100 | 누적 문장 수=1,077
[교육학] batch=13 | 추가 문장 수=77 | 누적 문장 수=1,154
[교육학] batch=14 | 추가 문장 수=119 | 누적 문장 수=1,273
[교육학] batch=15 | 추가 문장 수=120 | 누적 문장 수=1,393
중간 저장 완료 -> ../data/processed/ai\ai_370_temp_until_batch_15.csv
[교육학] batch=16 | 추가 문장 수=97 | 누적 문장 수=1,490
[교육학] batch=17 | 추가 문장 수=80 | 누적 문장 수=1,570
[교육학] batch=18 | 추가 문장 수=99 | 누적 문장 수=1,669
[교육학] ba

In [20]:
# 400: 자연과학
df_ai_400 = build_ai_topic_dataset_parallel(
    topic_code="400",
    target_n=TARGET_PER_TOPIC,
    save_every=SAVE_EVERY,
    batch_docs=BATCH_DOCS,
    max_workers=MAX_WORKERS
)


===== 자연과학 (400) AI 데이터 생성 시작 =====
[자연과학] batch=1 | 추가 문장 수=79 | 누적 문장 수=79
[자연과학] batch=3 | 추가 문장 수=80 | 누적 문장 수=159
[자연과학] batch=4 | 추가 문장 수=100 | 누적 문장 수=259
[자연과학] batch=2 | 추가 문장 수=137 | 누적 문장 수=396
[자연과학] batch=5 | 추가 문장 수=99 | 누적 문장 수=495
중간 저장 완료 -> ../data/processed/ai\ai_400_temp_until_batch_5.csv
[자연과학] batch=6 | 추가 문장 수=99 | 누적 문장 수=594
[자연과학] batch=7 | 추가 문장 수=140 | 누적 문장 수=734
[자연과학] batch=8 | 추가 문장 수=96 | 누적 문장 수=830
[자연과학] batch=9 | 추가 문장 수=99 | 누적 문장 수=929
[자연과학] batch=10 | 추가 문장 수=100 | 누적 문장 수=1,029
중간 저장 완료 -> ../data/processed/ai\ai_400_temp_until_batch_10.csv
[자연과학] batch=11 | 추가 문장 수=118 | 누적 문장 수=1,147
[자연과학] batch=12 | 추가 문장 수=78 | 누적 문장 수=1,225
[자연과학] batch=14 | 추가 문장 수=100 | 누적 문장 수=1,325
[자연과학] batch=13 | 추가 문장 수=99 | 누적 문장 수=1,424
[자연과학] batch=15 | 추가 문장 수=140 | 누적 문장 수=1,564
중간 저장 완료 -> ../data/processed/ai\ai_400_temp_until_batch_15.csv
[자연과학] batch=16 | 추가 문장 수=77 | 누적 문장 수=1,641
[자연과학] batch=17 | 추가 문장 수=73 | 누적 문장 수=1,714
[자연과학] batch=19 | 추가 문장 수=75

In [21]:
# 440: 천문학
df_ai_440 = build_ai_topic_dataset_parallel(
    topic_code="440",
    target_n=TARGET_PER_TOPIC,
    save_every=SAVE_EVERY,
    batch_docs=BATCH_DOCS,
    max_workers=MAX_WORKERS
)


===== 천문학 (440) AI 데이터 생성 시작 =====
[천문학] batch=4 | 추가 문장 수=80 | 누적 문장 수=80
[천문학] batch=3 | 추가 문장 수=100 | 누적 문장 수=180
[천문학] batch=1 | 추가 문장 수=120 | 누적 문장 수=300
[천문학] batch=2 | 추가 문장 수=137 | 누적 문장 수=437
[천문학] batch=5 | 추가 문장 수=99 | 누적 문장 수=536
중간 저장 완료 -> ../data/processed/ai\ai_440_temp_until_batch_5.csv
[천문학] batch=6 | 추가 문장 수=77 | 누적 문장 수=613
[천문학] batch=8 | 추가 문장 수=99 | 누적 문장 수=712
[천문학] batch=7 | 추가 문장 수=120 | 누적 문장 수=832
[천문학] batch=9 | 추가 문장 수=98 | 누적 문장 수=930
[천문학] batch=10 | 추가 문장 수=116 | 누적 문장 수=1,046
중간 저장 완료 -> ../data/processed/ai\ai_440_temp_until_batch_10.csv
[천문학] batch=12 | 추가 문장 수=100 | 누적 문장 수=1,146
[천문학] batch=11 | 추가 문장 수=119 | 누적 문장 수=1,265
[천문학] batch=13 | 추가 문장 수=79 | 누적 문장 수=1,344
[천문학] batch=15 | 추가 문장 수=78 | 누적 문장 수=1,422
[천문학] batch=14 | 추가 문장 수=134 | 누적 문장 수=1,556
중간 저장 완료 -> ../data/processed/ai\ai_440_temp_until_batch_15.csv
[천문학] batch=16 | 추가 문장 수=118 | 누적 문장 수=1,674
[천문학] batch=17 | 추가 문장 수=118 | 누적 문장 수=1,792
[천문학] batch=18 | 추가 문장 수=99 | 누적 문장 수=1,891

In [27]:
# =========================================================
# 8. 전체 통합 저장 (320 포함, 총 5만개)
# =========================================================

path_ai_320 = os.path.join(OUTPUT_DIR, "ai_320_final_10000.csv")
df_ai_320 = pd.read_csv(path_ai_320)

ai_all_df = pd.concat(
    [df_ai_320, df_ai_360, df_ai_370, df_ai_400, df_ai_440],
    axis=0
).reset_index(drop=True)

before_dedup = len(ai_all_df)
ai_all_df = ai_all_df.drop_duplicates(subset=["text"]).reset_index(drop=True)
after_dedup = len(ai_all_df)

print(f"중복 제거 전 문장 수: {before_dedup:,}")
print(f"중복 제거 후 문장 수: {after_dedup:,}")

if after_dedup < 50000:
    print("주의: 중복 제거 후 5만 개보다 적다.")
else:
    ai_all_df = ai_all_df.head(50000).copy()

ai_all_path = os.path.join(OUTPUT_DIR, "ai_5topics_final_50000.csv")
ai_all_df.to_csv(ai_all_path, index=False, encoding="utf-8-sig")

print(f"\n전체 통합 저장 완료 -> {ai_all_path}")
print(f"전체 문장 수: {len(ai_all_df):,}")

중복 제거 전 문장 수: 50,000
중복 제거 후 문장 수: 49,989
주의: 중복 제거 후 5만 개보다 적다.

전체 통합 저장 완료 -> ../data/processed/ai\ai_5topics_final_50000.csv
전체 문장 수: 49,989


In [35]:
import os
import time
import pandas as pd

FINAL_PATH = os.path.join(OUTPUT_DIR, "ai_5topics_final_50000.csv")

# 현재 파일 다시 읽기
ai_all_df = pd.read_csv(FINAL_PATH)

existing_texts = set(ai_all_df["text"].astype(str).tolist())
current_count_440 = (ai_all_df["topic_code"].astype(str) == "440").sum()
needed = 10000 - current_count_440

print("현재 전체 개수:", len(ai_all_df))
print("현재 440 개수:", current_count_440)
print("추가 필요 개수:", needed)

new_rows = []
extra_batch_num = 999999

topic_code = "440"
topic_name = topic_info[topic_code]["topic_name"]
topic_desc = topic_info[topic_code]["prompt_topic"]

while needed > 0:
    try:
        raw_text, meta = generate_batch_text(
            topic_name=topic_name,
            topic_desc=topic_desc,
            batch_docs=BATCH_DOCS
        )

        docs = split_generated_documents(raw_text)

        for doc_idx, doc in enumerate(docs, start=1):
            sentences = split_sentences_korean(doc)
            cleaned_sents = clean_ai_sentences(sentences)

            for sent in cleaned_sents:
                if sent not in existing_texts:
                    new_rows.append({
                        "text": sent,
                        "topic_code": topic_code,
                        "topic_name": topic_name,
                        "source": "ai",
                        "generation_style": meta["style"],
                        "length_spec": meta["length_spec"],
                        "temperature": meta["temperature"],
                        "batch_num": extra_batch_num,
                        "doc_in_batch": doc_idx,
                        "word_count": len(sent.split())
                    })
                    existing_texts.add(sent)
                    needed -= 1

                    if needed <= 0:
                        break
            if needed <= 0:
                break

        print("남은 부족분:", needed)
        extra_batch_num += 1
        time.sleep(0.8)

    except Exception as e:
        print("오류:", e)
        time.sleep(3)

if new_rows:
    ai_all_df = pd.concat([ai_all_df, pd.DataFrame(new_rows)], axis=0).reset_index(drop=True)
    ai_all_df = ai_all_df.drop_duplicates(subset=["text"]).reset_index(drop=True)

# 저장
ai_all_df.to_csv(FINAL_PATH, index=False, encoding="utf-8-sig")

# 검증
df_check = pd.read_csv(FINAL_PATH)
print("\n저장 후 전체 개수:", len(df_check))
print("저장 후 주제별 개수:")
print(df_check["topic_code"].astype(str).value_counts().sort_index())
print("저장 후 중복 개수:", df_check.duplicated(subset=["text"]).sum())

현재 전체 개수: 49989
현재 440 개수: 9989
추가 필요 개수: 11
남은 부족분: 0

저장 후 전체 개수: 50000
저장 후 주제별 개수:
topic_code
320    10000
360    10000
370    10000
400    10000
440    10000
Name: count, dtype: int64
저장 후 중복 개수: 0


In [36]:
import os
import pandas as pd

FINAL_PATH = os.path.join(OUTPUT_DIR, "ai_5topics_final_50000.csv")

# 불러오기
df_final = pd.read_csv(FINAL_PATH)

# =========================================================
# 1. 전체 개수 확인
# =========================================================
print("전체 개수:", len(df_final))

# =========================================================
# 2. 주제별 개수 확인
# =========================================================
print("\n주제별 개수:")
topic_counts = df_final["topic_code"].astype(str).value_counts().sort_index()
print(topic_counts)

# =========================================================
# 3. 중복 확인
# =========================================================
dup_count = df_final.duplicated(subset=["text"]).sum()
print("\n중복 개수:", dup_count)

# =========================================================
# 4. 데이터프레임 구조 확인
# =========================================================
print("\n컬럼:")
print(df_final.columns)

# =========================================================
# 5. 혹시 모를 이상치 체크 (선택)
# =========================================================
assert len(df_final) == 50000, "❌ 전체 개수 틀림"
assert all(topic_counts == 10000), "❌ 주제별 개수 틀림"
assert dup_count == 0, "❌ 중복 존재"

print("\n🔥 최종 데이터셋 정상 완료")

전체 개수: 50000

주제별 개수:
topic_code
320    10000
360    10000
370    10000
400    10000
440    10000
Name: count, dtype: int64

중복 개수: 0

컬럼:
Index(['text', 'topic_code', 'topic_name', 'source', 'generation_style',
       'length_spec', 'temperature', 'batch_num', 'doc_in_batch',
       'word_count'],
      dtype='object')

🔥 최종 데이터셋 정상 완료


## 최종 데이터 합치기

In [1]:
import os
import pandas as pd

# 1. 파일 경로
human_path = "../data/processed/human/human_5topics_50k.csv"
ai_path = "../data/processed/ai/ai_5topics_final_50000.csv"

# 2. 데이터 불러오기
human_all_df = pd.read_csv(human_path)
ai_all_df = pd.read_csv(ai_path)

print("HUMAN shape:", human_all_df.shape)
print("AI shape:", ai_all_df.shape)

# 3. source 컬럼 확인 및 통일
human_all_df["source"] = "human"
ai_all_df["source"] = "ai"

# 4. AI 데이터에 length_bin 없으면 생성
if "length_bin" not in ai_all_df.columns:
    bins = [5, 10, 15, 20, 30, 51]
    labels = ["5-9", "10-14", "15-19", "20-29", "30-50"]
    ai_all_df["length_bin"] = pd.cut(
        ai_all_df["word_count"],
        bins=bins,
        labels=labels,
        right=False,
        include_lowest=True
    )

# 5. 필요한 컬럼만 정리
required_cols = ["text", "word_count", "length_bin", "topic_code", "topic_name", "source"]

human_all_df = human_all_df[required_cols].copy()
ai_all_df = ai_all_df[required_cols].copy()

# topic_code 형식 통일
human_all_df["topic_code"] = human_all_df["topic_code"].astype(str)
ai_all_df["topic_code"] = ai_all_df["topic_code"].astype(str)

# 6. 최종 통합
final_df = pd.concat([human_all_df, ai_all_df], axis=0).reset_index(drop=True)

# 7. 저장
save_path = "../data/final/final_human_ai_100k.csv"
final_df.to_csv(save_path, index=False, encoding="utf-8-sig")

HUMAN shape: (50000, 6)
AI shape: (50000, 10)


In [46]:
# 8. 확인 출력
print("\n최종 저장 완료 ->", save_path)
print("최종 shape:", final_df.shape)


최종 저장 완료 -> ../data/final/final_human_ai_100k.csv
최종 shape: (100000, 6)


In [47]:
print("\n[source 분포]")
print(final_df["source"].value_counts())


[source 분포]
source
human    50000
ai       50000
Name: count, dtype: int64


In [48]:
print("\n[topic_name 분포]")
print(final_df["topic_name"].value_counts())


[topic_name 분포]
topic_name
경제학      20000
법률·법학    20000
교육학      20000
자연과학     20000
천문학      20000
Name: count, dtype: int64


In [3]:
# source x topic_name 분포
final_df.groupby(["source", "topic_name"]).size()

source  topic_name
ai      경제학           10000
        교육학           10000
        법률·법학         10000
        자연과학          10000
        천문학           10000
human   경제학           10000
        교육학           10000
        법률·법학         10000
        자연과학          10000
        천문학           10000
dtype: int64

In [5]:
final_df.groupby("source")["word_count"].describe()

,count,mean,std,min,25%,50%,75%,max
source,,,,,,,,
ai,50000.0,10.43092,2.593485,5.0,9.0,10.0,12.0,25.0
human,50000.0,13.47154,7.059064,5.0,8.0,12.0,17.0,50.0


In [51]:
print("\n전체 text 중복 수:", final_df["text"].duplicated().sum())


전체 text 중복 수: 0


In [4]:
final_df

,text,word_count,length_bin,topic_code,topic_name,source
0,그러면서 기술과 자본이 어우러져 만들어내는 생산력이 19세기 패권 경쟁에서 가장 중...,13,10-14,320,경제학,human
1,그렇다면 차를 살 때도 약간의 수고를 감수하고 바지런을 떠는 게 더 좋지 않을까.,13,10-14,320,경제학,human
2,그 마음으로 여러분만의 '사이드 프로젝트'를 시작하세요.,6,5-9,320,경제학,human
3,그러나 나는 입사해서 지금까지 매일 받는 봉사료를 모아서 생활비로 쓰고 있다.,11,10-14,320,경제학,human
4,"'사회운동노조주의'는 노조 스스로 정치적으로 강화돼야 한다고 하지만, 그것이 어떤 ...",11,10-14,320,경제학,human
...,...,...,...,...,...,...
99995,"행성은 별 주위를 공전하는 천체로, 스스로 빛을 내지 않는다는 점이 별과 다릅니다.",12,10-14,440,천문학,ai
99996,지구처럼 표면에 물이 존재할 수 있는 행성은 생명체가 살 수 있는 환경과도 연결되어...,15,15-19,440,천문학,ai
99997,예를 들어 화성은 과거에 물이 흐른 흔적이 발견되어 오랫동안 연구 대상이 되어 왔습니다.,13,10-14,440,천문학,ai
99998,"목성은 거대한 가스 행성으로, 강한 중력 때문에 수많은 위성을 거느립니다.",10,10-14,440,천문학,ai
